# RAG System with Metadata Support

A production-ready **Retrieval-Augmented Generation (RAG)** system that processes multiple documents and provides intelligent Q&A with comprehensive source tracking.

======= Key Features =======:
- **Multi-document support**: Processes all `.txt` files from documents directory
- **Rich metadata tracking**: Captures source files, file paths, and content previews
- **Google Gemini integration**: Uses Gemini embeddings and LLM for high-quality responses
- **Persistent vector storage**: Chroma database with automatic save/load functionality
- **Interactive Q&A session**: Command-line interface with system info and graceful exit
- **Error handling & logging**: Production-ready with comprehensive error management

======= Workflow =======:
1. Load multiple documents → split into chunks with metadata
2. Generate Gemini embeddings → persist in Chroma vector store
3. Interactive Q&A → retrieve relevant chunks + generate contextual answers
4. Display responses with source attribution and content previews

Perfect for document collections requiring intelligent

## import dependencies

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

import os
from typing import List, Dict, Any
from langchain.schema import Document
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

## global config

In [2]:
DOCUMENTS_DIR = "documents"
DB_DIR = "db"
PERSISTENT_DIR = "chroma_db_with_metadata"
EMBEDDING_MODEL = "models/gemini-embedding-001"
LLM_MODEL = "gemini-2.5-flash"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 50
RETRIEVAL_K = 3
SCORE_THRESHOLD = 0.2

## class init for RAG system

In [3]:
class RAGSystem:
    """RAG Q&A system for document collections."""

    def __init__(
            self,
            doc_dir: str = DOCUMENTS_DIR,
            db_dir: str = DB_DIR,
            embedding_model: str = EMBEDDING_MODEL,
            llm_model: str = LLM_MODEL,
            chunk_size: str = CHUNK_SIZE,
            chunk_overlap: str = CHUNK_OVERLAP,
            retrieval_k: int = RETRIEVAL_K,
            score_threshold: float = SCORE_THRESHOLD
    ):
        """Initialize the RAG system with configuration parameters."""
        self.doc_dir = os.path.join(os.getcwd(), doc_dir)
        self.persistent_dir = os.path.join(os.getcwd(), db_dir, PERSISTENT_DIR)
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.retrieval_k = retrieval_k
        self.score_threshold = score_threshold

        # init components
        self.embeddings = None
        self.vector_store = None
        self.qa_chain = None

        logger.info(f"RAG System initialized with doc_dir: {self.doc_dir}")

    def _validate_env(self) -> None:
        """Validate required environment variables and dependencies."""
        if not os.getenv('GOOGLE_API_KEY'):
            logger.error("GOOGLE_API_KEY environment variable is not set.")
            raise EnvironmentError("GOOGLE_API_KEY is required for RAG system.")
        
        if not os.path.exists(self.doc_dir):
            logger.error(f"Documents directory '{self.doc_dir}' does not exist.")
            raise FileNotFoundError(f"Documents directory '{self.doc_dir}' not found.")
        
    def _load_documents(self) -> List[Document]:
        """Load documents from the specified directory."""
        documents = []
        text_files = [file for file in os.listdir(self.doc_dir) if file.endswith('.txt')]
        if not text_files:
            logger.warning(f"No text files found in directory '{self.doc_dir}'.")
            raise FileNotFoundError(f"No text files found in '{self.doc_dir}'.")
        
        logger.info(f"found {len(text_files)} text files in '{self.doc_dir}'.")

        for file in text_files:
            try:
                file_path = os.path.join(self.doc_dir, file)
                loader = TextLoader(file_path=file_path)
                file_docs = loader.load()

                for doc in file_docs:
                    doc.metadata.update({
                        "source": file,
                        "file_path": file_path
                    })
                    documents.append(doc)
                logger.info(f"loaded document: {file}")
            except Exception as e:
                logger.error(f"Error loading document '{file}': {e}")
                continue

        return documents
    
    def _create_vector_store(self) -> None:
        """Create a vector store from the loaded documents."""

        self.embeddings = GoogleGenerativeAIEmbeddings(model=self.embedding_model)

        if os.path.exists(self.persistent_dir):
            logger.info(f"Loading existing vector store from '{self.persistent_dir}'")
            self.vector_store = Chroma(
                persist_directory=self.persistent_dir,
                embedding_function=self.embeddings
            )
        else:
            logger.info(f"Creating new vector store in '{self.persistent_dir}'")
            documents = self._load_documents()

            # split documents into chunks
            text_splitter = CharacterTextSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap
            )
            docs = text_splitter.split_documents(documents)
            logger.info(f"Split documents into {len(docs)} chunks.")

            # create vector store
            os.makedirs(os.path.dirname(self.persistent_dir), exist_ok=True)
            self.vector_store = Chroma.from_documents(
                documents=docs,
                embedding=self.embeddings,
                persist_directory=self.persistent_dir
            )
            logger.info(f"Vector store created and persisted.")

    def _setup_qa_chain(self) -> None:
        """Setup the question-answering chain."""
        llm = ChatGoogleGenerativeAI(model=self.llm_model, temperature=0.1)
        prompt_template = """
        Use the following pieces of context to answer the question at the end. 
        If you don't know the answer based on the context provided, just say that you don't know, don't try to make up an answer.
        
        Context:
        {context}
        
        Question: {question}
        
        Answer:
        """
        prompt = PromptTemplate(
            template=prompt_template,
            input_variables=['context', 'question']
        )
        retriever = self.vector_store.as_retriever(
            search_type='similarity_score_threshold',
            search_kwargs={
                "k": self.retrieval_k,
                "score_threshold": self.score_threshold
            }
        )
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={"prompt": prompt}
        )

        logger.info("Question-answering chain setup complete.")

    def initialize(self) -> None:
        """Initialize the RAG system components."""
        try:
            self._validate_env()
            self._load_documents()
            self._create_vector_store()
            self._setup_qa_chain()
            logger.info("RAG system initialized successfully.")
        except Exception as e:
            logger.error(f"Error during initialization: {str(e)}")
            raise RuntimeError(f"Failed to initialize RAG system: {str(e)}")
        
    def ask_question(self, question: str) -> Dict[str, Any]:
        """
        Ask a question and get an answer with source documents.
        
        Args:
            question: The question to ask
            
        Returns:
            Dict containing answer, sources, and metadata
        """
        if not self.qa_chain:
            raise RuntimeError("RAG system is not initialized. Call 'initialize()' first.")
        
        try:
            logger.info(f"Asking question: {question}")
            result = self.qa_chain.invoke({'query': question})
            response = {
                'question': question,
                'answer': result['result'],
                'sources': [],
            }

            # process source documents
            for doc in result.get('source_documents', []):
                source_info = {
                    'source': doc.metadata.get('source', 'unknown'),
                    'content_preview': doc.page_content[:200],  
                    'relevance_score': getattr(doc, 'relevance_score', 'N/A'),
                }
                response['sources'].append(source_info)
                logger.info(f"question answered with {len(response['sources'])} sources")

            return response
        except Exception as e:
            logger.error(f"Error during question answering: {str(e)}")
            return {
                "question": question,
                "answer": f"Sorry, I encountered an error: {str(e)}",
                "sources": []
            }
        
    def get_system_info(self) -> Dict[str, Any]:
        """Get information about the RAG system status."""
        info = {
            "documents_directory": self.doc_dir,
            "vector_store_exists": os.path.exists(self.persistent_dir),
            "system_initialized": self.qa_chain is not None,
            "configuration": {
                "embedding_model": self.embedding_model,
                "llm_model": self.llm_model,
                "chunk_size": self.chunk_size,
                "chunk_overlap": self.chunk_overlap,
                "retrieval_k": self.retrieval_k,
                "score_threshold": self.score_threshold
            }
        }
        
        if os.path.exists(self.doc_dir):
            text_files = [f for f in os.listdir(self.doc_dir) if f.endswith(".txt")]
            info["available_documents"] = text_files
            info["document_count"] = len(text_files)
        
        return info

In [4]:
def interactive_qa_session(rag_system: RAGSystem) -> None:
    """Run an interactive Q&A session."""
    print("\n" + "="*60)
    print("🤖 RAG Q&A System")
    print("="*60)
    print("Ask questions about your documents. Type 'quit', 'exit', or 'q' to stop.")
    print("Type 'info' to see system information.")
    print("="*60)

    while True:
        try:
            print("="*50)
            question = input("\n❓ Your question: ").strip()
            print(f"\nUser: {question}\n")
            
            if question.lower() in ['quit', 'exit', 'q']:
                print("\n👋 Goodbye!")
                break
                
            if question.lower() == 'info':
                info = rag_system.get_system_info()
                print(f"\n📊 System Information:")
                print(f"   Documents: {info['document_count']} files")
                print(f"   Vector Store: {'✅ Ready' if info['vector_store_exists'] else '❌ Not Found'}")
                print(f"   System: {'✅ Initialized' if info['system_initialized'] else '❌ Not Ready'}")
                print(f"   Configuration: {info['configuration']} \n")
                continue
                
            if not question:
                print("\nPlease enter a question: ")
                continue
                
            # Get answer
            response = rag_system.ask_question(question)
            
            print(f"\n💡 Answer: {response['answer']}")
            
            if response['sources']:
                print(f"\n📚 Sources ({len(response['sources'])} found):")
                for i, source in enumerate(response['sources'], 1):
                    print(f"   {i}. {source['source']}: {source['content_preview']}")
            else:
                print("\n📚 No source documents found.")
                
        except KeyboardInterrupt:
            print("\n\n👋 Session interrupted. Goodbye!")
            break
        except Exception as e:
            logger.error(f"Error in interactive session: {str(e)}")
            print(f"\n❌ Error: {str(e)}")

## settings for interactive Q&A session

In [5]:
RUN_INTERACTIVE_SESSION = True

In [6]:
def main():
    """Main function to run the production RAG system."""
    try:
        # Initialize the RAG system
        rag = RAGSystem()
        
        print("🚀 Initializing Production RAG System...")
        rag.initialize()
        
        # Show system info
        info = rag.get_system_info()
        print(f"\n✅ System ready with {info['document_count']} documents")
        
        # Start interactive session
        if RUN_INTERACTIVE_SESSION:
            interactive_qa_session(rag)
        else:
            sample_questions = [
                "Who is jiten parmar?"
            ]
            for question in sample_questions:
                print(f"\n❓ Asking: {question}")
                response = rag.ask_question(question)
                print(f"💡 Answer: {response['answer']}")
                if response['sources']:
                    print("📚 Sources:")
                    for source in response['sources']:
                        print(f"   - {source['source']}: {source['content_preview']}")
                else:
                    print("📚 No sources found.")
    except Exception as e:
        logger.error(f"Application error: {str(e)}")
        print(f"\n❌ Application error: {str(e)}")

In [7]:
main()

2025-08-03 17:51:50 - __main__ - INFO - RAG System initialized with doc_dir: /Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents
2025-08-03 17:51:50 - __main__ - INFO - found 2 text files in '/Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents'.
2025-08-03 17:51:50 - __main__ - INFO - loaded document: jiten-parmar.txt
2025-08-03 17:51:50 - __main__ - INFO - loaded document: lord_of_the_rings.txt
2025-08-03 17:51:50 - __main__ - INFO - Loading existing vector store from '/Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/db/chroma_db_with_metadata'
2025-08-03 17:51:50 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-08-03 17:51:50 - __main__ - INFO - Question-answering chain setup complete.
2025-08-03 17:51:50 - __main__ - INFO - RAG system initia

🚀 Initializing Production RAG System...

✅ System ready with 2 documents

🤖 RAG Q&A System
Ask questions about your documents. Type 'quit', 'exit', or 'q' to stop.
Type 'info' to see system information.

User: info


📊 System Information:
   Documents: 2 files
   Vector Store: ✅ Ready
   System: ✅ Initialized
   Configuration: {'embedding_model': 'models/gemini-embedding-001', 'llm_model': 'gemini-2.5-flash', 'chunk_size': 1000, 'chunk_overlap': 50, 'retrieval_k': 3, 'score_threshold': 0.2} 



2025-08-03 17:52:11 - __main__ - INFO - Asking question: who is jiten parmar



User: who is jiten parmar



2025-08-03 17:52:17 - __main__ - INFO - question answered with 1 sources
2025-08-03 17:52:17 - __main__ - INFO - question answered with 2 sources
2025-08-03 17:52:17 - __main__ - INFO - question answered with 3 sources



💡 Answer: Jiten Parmar is an AI/ML Engineer and Backend Developer with a strong focus on building production-ready ML systems and intelligent applications. He possesses expertise in backend development, CNN models, RAG pipelines, and generative AI solutions.

He has led significant projects, including "TestRAGic" (an AI-Powered QA Automation Platform) and "Dermalyse" (an AI-Powered Medical Diagnostic Tool). Jiten holds a BTech. in Computer Engineering from Shah & Anchor Kutchhi Engineering College. His technical experience includes virtual internships with Google for Developers AI/ML and Google Cloud services Generative AI, and a role as a Trainee Programmer at Chintan Systems Pvt. Ltd.

📚 Sources (3 found):
   1. jiten-parmar.txt: Jiten Parmar is an AI/ML Engineer and Backend Developer with a strong focus on building production-ready ML systems and intelligent applications. He possesses expertise in backend development and has 
   2. jiten-parmar.txt: Jiten has also led significant p

# response structure by qa_chain()

{
    "question": "Who is jiten parmar?",
    "answer": (
        "Jiten Parmar is an AI/ML Engineer and Backend Developer with a strong focus on building "
        "production-ready ML systems and intelligent applications. He possesses expertise in backend "
        "development and has hands-on experience with CNN models, RAG pipelines, and generative AI solutions."
    ),
    "sources": [
        {
            "source": "jiten-parmar.txt",
            "file_path": "/Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents/jiten-parmar.txt",
            "content_preview": "Jiten Parmar is an AI/ML Engineer and Backend Developer with a strong focus on building production-ready ML systems and intelligent applications. He possesses expertise in backend development and has hands-on experience with CNN models, RAG pipelines, and generative AI solutions. His technical skills include AI/ML and Data (AI Agents, Generative AI, TensorFlow, PyTorch, Scikit-learn, LangChain, CNN, RAG, Vector Databases, NLP, LLMs, AI APIs), Backend & Cloud (Python, Flask, Django, Node.js, JavaScript, TypeScript, GCP, AWS, Docker, Kubernetes), and Data & Tools (PostgreSQL, MongoDB, Version Control, RESTful APIs, Microservices)."
        },
        {
            "source": "jiten-parmar.txt",
            "file_path": "/Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents/jiten-parmar.txt",
            "content_preview": "Jiten has also led significant projects. \"TestRAGic,\" an AI-Powered QA Automation Platform, involved developing an intelligent test generation system using Python, LangChain, and OpenAI GPT-4 to convert video content into executable test cases, reducing manual test creation time by 75%. This project implemented a RAG pipeline with vector databases (FAISS/ChromaDB) and integrated Playwright for multi-browser automation. \"Dermalyse,\" an AI-Powered Medical Diagnostic Tool, saw him architecting a CNN model that achieved 90% accuracy in classifying 15 categories of skin diseases. The ML pipeline was engineered with TensorFlow for model training and inference optimization, resulting in a 35% reduction in misdiagnosis rates compared to baseline models in preliminary tests.\n\nHe holds a BTech. in Computer Engineering from Shah & Anchor Kutchhi Engineering College, securing a CGPA of 8.17 (2021-2025), and completed his HSC (12th) from SVKM's Mithibai College with 82.33% (2019-2021)."
        },
        {
            "source": "jiten-parmar.txt",
            "file_path": "/Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents/jiten-parmar.txt",
            "content_preview": "In his technical experience, Jiten completed virtual internships with Google for Developers AI/ML and Google Cloud services Generative AI through the All India Council For Technical Education. During the AI/ML internship (Oct-Dec 2024), he engineered over 3 production-ready ML models using TensorFlow and AutoML on Google Cloud Platform, learning fundamental concepts of image classification, object detection, and product image search. In the Generative AI internship (July-Sept 2024), he built over 4 generative AI applications using Vertex AI and Gemini models, gaining hands-on experience with GCP services for generative AI, including language models and image generation. As a Trainee Programmer at Chintan Systems Pvt. Ltd. (Dec 2023-Jan 2024), he architected comprehensive API documentation for over 30 endpoints, reducing integration time by 45% by defining payload specifications and response schemas."
        }
    ]
}